# 02. Face Attribute - face_attr_resnet_v1_18

| 항목 | 내용 |
|---|---|
| 대상 모델 | `face_attr_resnet_v1_18.dxnn` |
| 카테고리 | **Face Attribute** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 속성마다 따로 softmax 를 걸어야 하는 이유를 한 번에 거는 방식과 비교해 설명할 수 있다.
2. 속성당 점수가 두 개인 출력에서 '그렇다' 확률을 직접 계산할 수 있다.
3. 임계값을 바꿀 때 결과 개수가 어떻게 달라지는지 실행해 확인할 수 있다.
4. `run()` 이 돌려준 값이 NPU 계산 결과가 아님을 Task 구성으로 설명할 수 있다.

## 1. Face Attribute (얼굴 속성)

**얼굴 사진 한 장에서 정해진 속성 여러 개를 각각 참·거짓으로 판정하는 작업**

판정 대상이 40가지라면 40개의 독립된 예·아니오 질문을 한 번에 푸는 셈이다. 속성들은 서로 경쟁하지
않으므로 여러 개가 동시에 참일 수 있고, 하나도 참이 아닐 수도 있다.

**출력의 성격**

항목별 독립 판정이라 답의 개수가 정해져 있지 않다. 좌표가 없어 얼굴이 사진 어디에 있는지는 알려 주지
않으며, 얼굴이 이미 화면을 채우고 있어야 판정이 성립한다.

**활용 사례**

- 얼굴 사진 자동 태깅
- 안경·모자 착용 여부 확인
- 검출·정렬 뒤에 붙이는 속성 판정 단계

**포함되지 않는 작업**

- 사진에서 얼굴 위치 찾기 (Face Detection)
- 누구인지 대조 (Face Embedding)
- 하나의 정답만 고르기 (Image Classification)

## 2. 전체 파이프라인

```text
sample_person_a1.jpg  1024×1024 BGR
  → 전처리         BGR→RGB · 178×218 직접 축소 · uint8 유지
  → run()          입력 Tensor 하나를 NPU 에 전달
  → npu_0          218  [1, 80]  float32   (내부 전달, Public 아님)
  → cpu_0          220  [1, 40, 2]  float32  (Public Output)
  → 후처리         속성마다 2개씩 softmax → 임계값 0.5 → 확률 내림차순
  → Result         통과한 속성만 ClassificationResult (이름까지 채움)
  → 시각화         왼쪽 위에 속성 목록을 글자로 얹음
```

![02. Face Attribute 전체 파이프라인](resources/02_Face_Attribute_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `input.1` |
| Shape / Dtype | `[1, 218, 178, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**다른 분류 모델과의 차이점**

- **정사각이 아닌 입력** — 178×218 로 세로가 더 길다. 얼굴이 세로로 긴 형태이고 학습에 쓴 사진 규격을
  그대로 따랐다
- **얼굴이 이미 잘려 있어야 함** — 이 모델에는 얼굴을 찾는 단계가 없다. 얼굴이 작게 찍힌 사진을 넣으면
  배경까지 뭉개져 들어가 판정이 의미를 잃는다
- **사진 입력만 받음** — 영상·카메라 입력은 실행 진입점에서 거부된다. 영상에 쓰려면 앞에 검출과 잘라내기
  단계를 붙여야 한다

### 3.2 출력

이 모델은 `.dxnn` 하나 안에 Task 가 둘이다. `run()` 이 돌려주는 값은 **NPU 계산 결과가 아니라 CPU Task
가 모양을 바꾼 뒤의 값**이다.

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `218` | `[1, 80]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `220` | `[1, 40, 2]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

<br>

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 40 | 속성 개수. 각 자리가 속성 하나 |
| 2 | 2 | 그 속성의 '아니다' 점수와 '그렇다' 점수 |

`float32` 하나가 4바이트이므로 `1 × 40 × 2` 는 320 바이트다.

**해당 모델은 속성당 점수가 두 개이다.** 40개 전체에 softmax 를 한 번 걸면 40개 확률의 합이
1 이 되어 "안경도 썼고 웃고도 있다"를 표현할 수 없다. 속성들은 경쟁 관계가 아니므로 **각 줄의 두 점수에만
따로** softmax 를 건다.

```text
속성 0  :  [ 아니다 점수 , 그렇다 점수 ]  → softmax → [ p_no , p_yes ]   p_no + p_yes = 1
속성 1  :  [ 아니다 점수 , 그렇다 점수 ]  → softmax → [ p_no , p_yes ]
   ⋮
속성 39 :  [ 아니다 점수 , 그렇다 점수 ]
```

점수 개수로 방식을 갈라낼 수 있다. 속성 수의 **두 배**면 두 개씩 짝지어 softmax 를 걸고, 속성 수와
**같으면** 각각 시그모이드를 건다. 두 방식은 학습 방법이 다르므로 바꿔 쓰면 확률이 통째로 틀리는데, 값은
0~1 사이라 그럴듯해 보인다.

**CelebA 속성 40개:**

| index | 속성 | index | 속성 | index | 속성 | index | 속성 |
|---:|---|---:|---|---:|---|---:|---|
| 0 | 5 o'Clock Shadow | 10 | Blurry | 20 | Male | 30 | Sideburns |
| 1 | Arched Eyebrows | 11 | Brown Hair | 21 | Mouth Slightly Open | 31 | Smiling |
| 2 | Attractive | 12 | Bushy Eyebrows | 22 | Mustache | 32 | Straight Hair |
| 3 | Bags Under Eyes | 13 | Chubby | 23 | Narrow Eyes | 33 | Wavy Hair |
| 4 | Bald | 14 | Double Chin | 24 | No Beard | 34 | Wearing Earrings |
| 5 | Bangs | 15 | Eyeglasses | 25 | Oval Face | 35 | Wearing Hat |
| 6 | Big Lips | 16 | Goatee | 26 | Pale Skin | 36 | Wearing Lipstick |
| 7 | Big Nose | 17 | Gray Hair | 27 | Pointy Nose | 37 | Wearing Necklace |
| 8 | Black Hair | 18 | Heavy Makeup | 28 | Receding Hairline | 38 | Wearing Necktie |
| 9 | Blond Hair | 19 | High Cheekbones | 29 | Rosy Cheeks | 39 | Young |


#### 출력을 사람이 읽는 값으로 바꾸기

값이 둘뿐인 softmax 이므로 결과도 둘이고 합이 1 이다. 그중 '그렇다' 쪽을 그 속성의 확률로 쓴다.

```text
m = max(z_no, z_yes)
p_yes = exp(z_yes - m) / ( exp(z_no - m) + exp(z_yes - m) )
```

이번 실행의 20번 속성(`Male`)으로 계산하면 두 점수가 `[-0.5592, 0.5664]` 다. `m = 0.5664` 를 빼면
`[-1.1256, 0]` 이고 `exp` 는 `[0.3244, 1]` 이므로 `p_yes = 1 / 1.3244 = 0.7550` 즉 75.5% 다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. Task 두 개가
`npu_0` → `cpu_0` 로 이어지고 `220` 만 model output 으로 표시되는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "face_attr_resnet_v1_18.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 CelebA 속성 이름 40개를 읽는다. 이후 셀은 여기서 만든
이름을 그대로 쓴다.

In [ ]:
import sys
import time
from dataclasses import dataclass

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

IMAGE_PATH = DX_APP / "sample" / "img/sample_person_a1.jpg"

# dx-all-suite 는 읽기 전용 원본이다. 바이트코드 캐시를 남기지 않도록 먼저 꺼 둔다.
sys.dont_write_bytecode = True
sys.path.insert(0, str(DX_APP / "src" / "python_example"))
from common.processors.attribute_postprocessor import CELEBA_40_LABELS  # noqa: E402

print("attributes", len(CELEBA_40_LABELS), "/ index 20 =", CELEBA_40_LABELS[20])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 이 모델의 후처리는
  되돌릴 좌표가 없어 받기만 하고 읽지 않는다
- `ClassificationResult` — 후처리가 만들어 시각화로 넘기는 결과 객체. 속성 이름까지 채워진다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class ClassificationResult:
    """후처리가 만들어 시각화로 넘기는 결과. 원본 대응: common/base/i_processor.py"""

    class_id: int = 0
    class_name: str = ""
    confidence: float = 0.0


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 출력이 `220` 하나로만 보이는 것이 CPU
Task 출력만 밖으로 나온다는 뜻이다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

1024×1024 BGR 원본을 RGB 178×218 `uint8` 로 바꾼다. 비율을 맞추는 여백을 넣지 않으므로 정사각 원본이
세로로 늘어난 형태가 되고, 그 축소 비율을 Context 에 축별로 적어 둔다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 178×218, 가로세로가 다름
- dtype — `uint8` 유지

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 얼굴이 화면을 채우고 있어야 판정이 성립한다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}")

#### 전처리 함수 정의

- `face_attr_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `face_attr_preprocessor_process` — 이미지 한 장을 입력 Tensor 와 Context 로 변환

In [ ]:
def face_attr_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def face_attr_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    # 비율을 맞추는 여백이 없으므로 축마다 축소 비율이 다르다. pad 는 0 으로 남는다.
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. 원본이 정사각인데 `scale_x` 와 `scale_y` 가 다르게 나오는 것이 세로로
늘어났다는 증거이며, 아래 그림에서 얼굴이 길어진 것으로 확인된다.

In [ ]:
preprocessor_state = face_attr_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = face_attr_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"scale_x={ctx.scale_x:.4f} scale_y={ctx.scale_y:.4f} pad=({ctx.pad_x}, {ctx.pad_y})")

notebook_show_bgr(cv2.cvtColor(input_tensor, cv2.COLOR_RGB2BGR),
                  f"Preprocessed {INPUT_WIDTH}x{INPUT_HEIGHT} - stretched", figsize=(4, 5))

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 NPU 에 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복
비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- shape — `[1, 40, 2]` 로 속성마다 점수가 두 개
- 점수 부호 — 음수가 섞여 있어 확률이 아니다
- 한 줄의 두 점수 — 대소가 뒤집히면 그 속성의 판정도 뒤집힌다

In [ ]:
raw = np.asarray(outputs[0])
notebook_describe_array("raw output", raw)

pairs = raw.reshape(-1, 2)
print(f"pairs shape = {pairs.shape}")
for index in (20, 24, 31):
    print(f"  [{index:2d}] {CELEBA_40_LABELS[index]:22s} no={pairs[index][0]:8.4f} "
          f"yes={pairs[index][1]:8.4f}")

### 4.4 후처리

속성마다 두 점수에 softmax 를 걸어 '그렇다' 확률을 얻고, 임계값을 넘는 속성만 남겨 확률 내림차순으로
정렬한다. Top-K 가 아니라 임계값 방식이므로 결과 개수가 정해져 있지 않다 — 하나도 없을 수도 있고 절반이
넘을 수도 있다.

**임계값 0.5 는 후처리 코드의 기본값이다.** 이 모델은 설정 파일이 비어 있어 값을 바꿀 자리가 없고 코드가
들고 있는 기본값이 그대로 쓰인다. 낮추면 결과가 늘면서 틀린 것도 늘고, 높이면 확실한 것만 남는다.

01번과 달리 **후처리가 속성 이름을 직접 채운다.** 임계값을 넘은 속성만 골라 내보내므로 어느 번호가
남았는지를 시각화 단계에서 다시 계산할 필요가 없다.

#### 후처리 함수 정의

- `face_attr_postprocessor_create` — 임계값과 이름 배열을 state 에 담음
- `face_attr_softmax_pairs` — 각 줄의 두 점수에 softmax 를 걸어 '그렇다' 확률만 반환
- `face_attr_postprocessor_process` — 임계값 통과 속성을 확률 내림차순 Result 로 생성

In [ ]:
def face_attr_postprocessor_create(threshold: float = 0.5, labels: list = None) -> dict:
    """고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"threshold": threshold, "labels": labels or CELEBA_40_LABELS}


def face_attr_softmax_pairs(logits: np.ndarray) -> np.ndarray:
    """[N, 2] 점수의 줄마다 softmax 를 걸어 '그렇다' 확률만 돌려준다.

    원본 대응: AttributePostprocessor.process() 의 CelebA 분기
    """
    exp = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp[:, 1] / exp.sum(axis=1)


def face_attr_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    output = np.asarray(outputs[0])
    probabilities = face_attr_softmax_pairs(output.reshape(-1, 2))

    results = [
        ClassificationResult(class_id=int(index), confidence=float(probability),
                             class_name=state["labels"][index])
        for index, probability in enumerate(probabilities)
        if probability > state["threshold"]
    ]
    results.sort(key=lambda result: result.confidence, reverse=True)
    return results

#### 후처리 실행

확인할 것:

- 결과 개수 — 임계값을 넘은 속성만 남으므로 40개보다 훨씬 적다
- `class_name` — 01번과 달리 후처리가 채워 넣는다
- 임계값을 바꾼 결과 — 놓치는 것과 잘못 잡는 것의 맞바꿈이 개수로 드러난다

In [ ]:
postprocessor_state = face_attr_postprocessor_create()
results = face_attr_postprocessor_process(postprocessor_state, outputs, ctx)

print(f"result count = {len(results)} / 40 (threshold={postprocessor_state['threshold']})")
for rank, res in enumerate(results, start=1):
    print(f"  #{rank} class_id={res.class_id:2d} confidence={res.confidence:.6f} "
          f"class_name={res.class_name!r}")

for threshold in (0.3, 0.5, 0.7, 0.9):
    state = face_attr_postprocessor_create(threshold=threshold)
    print(f"threshold={threshold} → {len(face_attr_postprocessor_process(state, outputs, ctx))}개")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 왼쪽 위에 속성 목록을 글자로 얹는다. 후처리가 이름을 이미 채웠으므로 번호를 다시
해석하지 않는다. 확률 0.7 을 넘는 속성은 초록, 그 아래는 노란빛으로 그려 확신의 정도를 색으로 나눈다.
그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def face_attr_visualizer_visualize(image: np.ndarray, results: list) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()
    if not results:
        return output

    height, width = output.shape[:2]
    line_h, pad = 22, 8
    box_w = min(width - 20, 320)
    box_h = pad * 2 + (len(results) + 1) * line_h

    overlay = output.copy()
    cv2.rectangle(overlay, (10, 10), (10 + box_w, 10 + box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, output, 0.45, 0, output)

    y_offset = 10 + pad + line_h
    cv2.putText(output, "[Person Attributes]", (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (0, 255, 255), 1, cv2.LINE_AA)
    y_offset += line_h

    for res in results:
        text = f"{res.class_name}: {res.confidence * 100:.0f}%"
        color = (0, 255, 0) if res.confidence > 0.7 else (0, 200, 200)
        cv2.putText(output, text, (25, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.48, color, 1,
                    cv2.LINE_AA)
        y_offset += line_h

    return output

#### 결과 시각화

최종 결과를 inline 으로 표시한다. 글자가 그려지는 대상은 178×218 로 늘린 입력이 아니라 1024×1024
원본이다.

In [ ]:
output_image = face_attr_visualizer_visualize(image, results)
notebook_show_bgr(output_image, "Face attributes over threshold", figsize=(9, 9))